# Prompt Engineering — NormaBot RAG

## Introducción

NormaBot utiliza un pipeline **Corrective RAG** con tres pasos:

1. **Retrieve** — recupera documentos de ChromaDB
2. **Grade** — filtra cuáles son relevantes para la pregunta (Ollama Qwen 2.5 3B)
3. **Generate** — genera la respuesta con citas legales (Bedrock Nova Lite)

Cada paso tiene un **prompt** que controla el comportamiento del LLM. Este notebook documenta el proceso de iteración sobre los prompts:

- Medimos el baseline (prompts actuales)
- Creamos 2-3 variantes con técnicas distintas
- Comparamos con métricas fijas
- Decidimos qué variante se queda y por qué

### Prompts analizados

| Prompt | LLM | Métrica principal |
|--------|-----|------------------|
| `GRADING_PROMPT` | Qwen 2.5 3B (Ollama) | Precision / Recall |
| `GENERATE_PROMPT` | Nova Lite (Bedrock) | Faithfulness / Answer Relevancy (RAGAS) |

## 1. Prompt de Grading

### ¿Qué hace?

Recibe un documento recuperado de ChromaDB y la pregunta del usuario.
Decide si el documento contiene información útil para responder — devuelve `si` o `no`.

### ¿Por qué importa?

Un grader demasiado **estricto** (recall bajo) descarta documentos útiles → el generador tiene menos contexto.
Un grader demasiado **permisivo** (precision baja) pasa basura al generador → riesgo de alucinaciones.

En un sistema legal como NormaBot, **el recall es prioritario**: preferimos pasar un documento dudoso antes de perder una cita legal relevante.

### Prompt v0 — Zero-shot (actual en producción)

```
Dado el siguiente documento y la pregunta,
¿el documento contiene información útil para responder la pregunta?

Documento: {document}
Pregunta: {query}

Responde solo con "si" o "no":
```

**Técnica**: Zero-shot — sin ejemplos, instrucción directa.

In [1]:
import json
from pathlib import Path

RESULTS_PATH = Path("../../eval/grading_results.json")

with open(RESULTS_PATH, encoding="utf-8") as f:
    results = json.load(f)

print("Variantes disponibles:", list(results.keys()))

Variantes disponibles: ['v0', 'v1', 'v2']


In [2]:
# Métricas baseline (v0)
v0 = results["v0"]["metrics"]
print("=== Baseline — Prompt v0 (Zero-shot) ===")
print(f"  Precision : {v0['precision']:.4f}")
print(f"  Recall    : {v0['recall']:.4f}")
print(f"  Accuracy  : {v0['accuracy']:.4f}")
print(f"  TP={v0['tp']}  FP={v0['fp']}  FN={v0['fn']}  Total={v0['n_total']}")

=== Baseline — Prompt v0 (Zero-shot) ===
  Precision : 1.0000
  Recall    : 0.5455
  Accuracy  : 0.7917
  TP=6  FP=0  FN=5  Total=24


### Análisis del baseline

| Métrica | Valor | Interpretación |
|---------|-------|----------------|
| Precision | 1.000 | Nunca acepta documentos irrelevantes (0 falsos positivos) |
| Recall | 0.545 | Descarta casi la mitad de los documentos relevantes (5 falsos negativos) |
| Accuracy | 0.792 | Acierta en 19 de 24 casos |

**Problema identificado**: El prompt v0 es demasiado conservador. Descarta documentos útiles, reduciendo el contexto disponible para el generador.

## 2. Variante v1 — Few-shot

**Técnica**: Añadir 2 ejemplos al prompt (1 relevante + 1 irrelevante) para guiar al modelo.

**Hipótesis**: Al ver ejemplos concretos, el modelo entenderá mejor qué significa "útil para responder".

```
Tu tarea es decidir si un documento contiene información útil para responder una pregunta.

Ejemplos:

Pregunta: ¿Qué prácticas de IA están prohibidas?
Documento: Artículo 5. Quedan prohibidas las siguientes prácticas de IA...
Respuesta: si

Pregunta: ¿Qué prácticas de IA están prohibidas?
Documento: Artículo 9. Se establecerá un sistema de gestión de riesgos...
Respuesta: no

Ahora evalúa:

Pregunta: {query}
Documento: {document}

Responde solo con "si" o "no":
```

In [3]:
v1 = results["v1"]["metrics"]
print("=== Variante v1 (Few-shot) ===")
print(f"  Precision : {v1['precision']:.4f}")
print(f"  Recall    : {v1['recall']:.4f}")
print(f"  Accuracy  : {v1['accuracy']:.4f}")
print(f"  TP={v1['tp']}  FP={v1['fp']}  FN={v1['fn']}  Total={v1['n_total']}")

=== Variante v1 (Few-shot) ===
  Precision : 1.0000
  Recall    : 0.5455
  Accuracy  : 0.7917
  TP=6  FP=0  FN=5  Total=24


## 3. Variante v2 — Chain of Thought

**Técnica**: Pedir al modelo que razone antes de decidir.

**Hipótesis**: Al forzar al modelo a identificar primero el tema del documento, mejora su capacidad de relacionarlo con la pregunta.

```
Dado el siguiente documento y la pregunta, determina si el documento contiene
información útil para responder la pregunta.

Documento: {document}
Pregunta: {query}

Primero identifica de qué trata el documento en una frase.
Luego decide si esa información ayuda a responder la pregunta.
Termina tu respuesta con "Veredicto: si" o "Veredicto: no".
```

**Nota técnica**: Esta variante requiere `num_predict=150` en lugar de 10, ya que el modelo necesita generar texto de razonamiento antes del veredicto.

In [4]:
v2 = results["v2"]["metrics"]
print("=== Variante v2 (Chain of Thought) ===")
print(f"  Precision : {v2['precision']:.4f}")
print(f"  Recall    : {v2['recall']:.4f}")
print(f"  Accuracy  : {v2['accuracy']:.4f}")
print(f"  TP={v2['tp']}  FP={v2['fp']}  FN={v2['fn']}  Total={v2['n_total']}")

=== Variante v2 (Chain of Thought) ===
  Precision : 1.0000
  Recall    : 0.5455
  Accuracy  : 0.7917
  TP=6  FP=0  FN=5  Total=24


## 4. Tabla comparativa

In [5]:
print(f"{'Métrica':<15} {'v0 (zero-shot)':>15} {'v1 (few-shot)':>15} {'v2 (CoT)':>15}")
print("-" * 62)
for metric in ["precision", "recall", "accuracy"]:
    print(f"{metric:<15} {results['v0']['metrics'][metric]:>15.4f} {results['v1']['metrics'][metric]:>15.4f} {results['v2']['metrics'][metric]:>15.4f}")

print()
print("Casos que fallan por variante:")
for variant in ["v0", "v1", "v2"]:
    fallos = [i+1 for i, r in enumerate(results[variant]["detail"]) if not r["correct"]]
    print(f"  {variant}: pares {fallos}")

Métrica          v0 (zero-shot)   v1 (few-shot)        v2 (CoT)
--------------------------------------------------------------
precision                1.0000          1.0000          1.0000
recall                   0.5455          0.5455          0.5455
accuracy                 0.7917          0.7917          0.7917

Casos que fallan por variante:
  v0: pares [1, 7, 11, 17, 21]
  v1: pares [7, 11, 15, 17, 21]
  v2: pares [7, 9, 11, 13, 21]


## 5. Análisis de errores

Los tres prompts obtienen el mismo recall (0.545) pero **fallan en casos distintos**:

| Par | Pregunta | Documento | v0 | v1 | v2 |
|-----|----------|-----------|----|----|----|
| 7  | ¿Qué multas prevé el EU AI Act? | Art. 71 (infracciones) | ❌ | ❌ | ❌ |
| 11 | ¿Requisitos de gobernanza de datos? | Art. 10 (conjuntos de datos) | ❌ | ❌ | ❌ |
| 21 | ¿Transparencia para riesgo limitado? | Art. 52 (informar a usuarios) | ❌ | ❌ | ❌ |

### Patrón de fallo

Los casos que fallan consistentemente comparten una característica: el documento usa **vocabulario indirecto** respecto a la pregunta:

- "multas" (pregunta) vs "infracciones" (documento)
- "gobernanza de datos" (pregunta) vs "conjuntos de datos de entrenamiento" (documento)  
- "riesgo limitado" (pregunta) vs el Art. 52 que no menciona explícitamente esa categoría

### Conclusión del análisis

El problema no está en el prompt sino en la **capacidad del modelo**: Qwen 2.5 3B no tiene suficiente comprensión semántica para relacionar términos sinónimos en contexto legal. Cambiar el prompt no resuelve esta limitación.

## 6. Decisión final

### Qué variante se queda

**Se mantiene v0 (Zero-shot)** por las siguientes razones:

1. **Ninguna variante mejoró el recall**: las tres obtienen recall=0.545 con 5 falsos negativos
2. **v1 intercambió errores**: arregló el par 1 pero rompió el par 15 — resultado neto igual
3. **v2 empeoró en casos fáciles**: con Chain of Thought, el modelo falló en pares (9, 13) que v0 resolvía correctamente
4. **Coste computacional**: v2 genera ~15x más tokens por llamada sin beneficio en métricas

### Recomendación para el equipo

El recall bajo (0.545) es una **limitación del modelo 3B**, no del prompt. Las opciones para mejorarlo son:

- Aumentar `k` en el retriever (recuperar más documentos) para compensar los falsos negativos del grader
- Usar un modelo más grande para grading (mayor coste de inferencia)
- Ajustar el umbral del fallback por score (`_grade_by_score`) como complemento al grading LLM

Estas decisiones afectan `src/rag/main.py` y deben consensuarse con el equipo.